In [2]:
import pandas as pd
from deep_translator import GoogleTranslator
from tqdm import tqdm
import time

# Load the English data
df = pd.read_csv(r'C:\Users\suki\Desktop\PFA2\PFA_Reclamations\ai-service\notebooks\data\processed\complaints_english.csv')
print(f"Loaded {len(df)} complaints")

C:\Users\suki\AppData\Local\Temp\ipykernel_16504\3121494642.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


Loaded 2563 complaints


In [4]:
# Take 200 examples per category to start (1400 total)
SAMPLE_SIZE = 200

df_sample = df.groupby('category').apply(
    lambda x: x.sample(min(len(x), SAMPLE_SIZE), random_state=42)
).reset_index(drop=True)

print(f"Sample size: {len(df_sample)}")

# Truncate very long texts to make translation faster
df_sample['text'] = df_sample['text'].str[:1000]

Sample size: 1112


C:\Users\suki\AppData\Local\Temp\ipykernel_16504\2550087429.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_sample = df.groupby('category').apply(


In [5]:
def translate_text(text):
    """
    Translates a text from English to French.
    Includes error handling and small delays.
    """
    try:
        if pd.isna(text) or len(str(text)) < 10:
            return None
        # Truncate to 1000 characters for safety (Google Translate limit is 5000)
        text = str(text)[:1000]
        translated = GoogleTranslator(source='en', target='fr').translate(text)
        time.sleep(0.5)  # small delay to avoid rate limits
        return translated
    except Exception as e:
        print(f"Error: {e}")
        return None

# Use tqdm to show progress bar
tqdm.pandas()

print("Translating... this will take a while (around 30-60 minutes)")
df_sample['text_fr'] = df_sample['text'].progress_apply(translate_text)

# Save progress (in case something crashes)
df_sample.to_csv('../data/processed/complaints_translated.csv', index=False)
print("Translation done and saved!")

Translating... this will take a while (around 30-60 minutes)


 19%|██████████████▍                                                              | 208/1112 [06:52<1:47:14,  7.12s/it]

Error: HTTPSConnectionPool(host='translate.google.com', port=443): Read timed out. (read timeout=None)


 22%|████████████████▉                                                            | 245/1112 [08:52<2:01:34,  8.41s/it]

Error: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))


...

Error: HTTPSConnectionPool(host='translate.google.com', port=443): Max retries exceeded with url: /m?tl=fr&sl=en&q=reported+identity+theft+bureau+requested+freeze+went+step+ftc+resolve+use+idenity+completely+taken+unemployment+claim+account+online+payment+even+refuse+one+ex+finance+stole+information+girlfriend+took+advantage+mental+illness+took+everything+dont+know+month+thing+getting+worst+even+sends+fake+letter+requesting+money+please+help (Caused by ConnectTimeoutError(<HTTPSConnection(host='translate.google.com', port=443) at 0x26a1e8abed0>, 'Connection to translate.google.com timed out. (connect timeout=None)'))


 56%|████████████████████████████████████████████▎                                  | 623/1112 [22:37<36:52,  4.52s/it]

Error: HTTPSConnectionPool(host='translate.google.com', port=443): Max retries exceeded with url: /m?tl=fr&sl=en&q=cashapp+username+scammed+dollar+know+bank+last+debit+card+also+lost+fake+cashapp+representative+hacked+phone+stole+account+app+know+person+phone+number+email (Caused by SSLError(SSLEOFError(8, 'EOF occurred in violation of protocol (_ssl.c:992)')))


100%|██████████████████████████████████████████████████████████████████████████████| 1112/1112 [36:51<00:00,  1.99s/it]


OSError: Cannot save file into a non-existent directory: '..\data\processed'

In [6]:
import os
import pandas as pd

# Créer le dossier
os.makedirs('../data/processed/', exist_ok=True)

# Sauvegarder le DataFrame (les traductions sont déjà dans df_sample)
df_sample.to_csv('../data/processed/complaints_translated.csv', index=False)

print(f"✔ Sauvegardé {len(df_sample)} traductions dans '../data/processed/complaints_translated.csv'")

✔ Sauvegardé 1112 traductions dans '../data/processed/complaints_translated.csv'


In [7]:
# Afficher un exemple
print("Texte original:", df_sample['text'].iloc[0][:100])
print("Traduction:", df_sample['text_fr'].iloc[0][:100])

Texte original: duplicate charge come original unemployment card followed step report fraudulent charge cancellation
Traduction: les frais en double viennent la carte de chômage originale a suivi l'étape rapporter la carte d'annu


In [8]:
# Remove failed translations
df_clean = df_sample[df_sample['text_fr'].notna()].copy()

# Final dataset
final_df = df_clean[['text_fr', 'category', 'priority']].copy()
final_df.columns = ['texte', 'categorie', 'priorite']

# Translate category names to French
category_map = {
    'late_delivery': 'retard_livraison',
    'broken_product': 'produit_casse',
    'wrong_item': 'erreur_picking',
    'missing_item': 'article_manquant',
    'poor_quality': 'mauvaise_qualite',
    'transport_problem': 'probleme_transport',
    'admin_error': 'erreur_administrative'
}

priority_map = {
    'critical': 'critique',
    'high': 'elevee',
    'medium': 'moyenne',
    'low': 'faible'
}

final_df['categorie'] = final_df['categorie'].map(category_map)
final_df['priorite'] = final_df['priorite'].map(priority_map)

print(final_df.head())
print(f"\nFinal size: {len(final_df)}")
print(f"\nCategory distribution:")
print(final_df['categorie'].value_counts())
print(f"\nPriority distribution:")
print(final_df['priorite'].value_counts())

                                               texte              categorie  \
0  les frais en double viennent la carte de chôma...  erreur_administrative   
1  contacté à plusieurs reprises l'entreprise exi...  erreur_administrative   
2  déposé litige poursuite visa charge compte fai...  erreur_administrative   
3  soumettre une plainte crédit extrêmement affec...  erreur_administrative   
4  transaction de crédit à la consommation entrée...  erreur_administrative   

  priorite  
0  moyenne  
1   elevee  
2  moyenne  
3  moyenne  
4  moyenne  

Final size: 1108

Category distribution:
categorie
retard_livraison         200
probleme_transport       200
erreur_picking           200
produit_casse            198
mauvaise_qualite         198
erreur_administrative    100
article_manquant          12
Name: count, dtype: int64

Priority distribution:
priorite
moyenne     439
elevee      369
faible      155
critique    145
Name: count, dtype: int64


In [9]:
# Save the final dataset ready for training
final_df.to_csv('../data/processed/reclamations_dataset_final.csv', 
                index=False, encoding='utf-8')

print("✅ FINAL DATASET SAVED!")
print(f"Location: data/processed/reclamations_dataset_final.csv")
print(f"Size: {len(final_df)} reclamations")

✅ FINAL DATASET SAVED!
Location: data/processed/reclamations_dataset_final.csv
Size: 1108 reclamations
